# 救急車 乗り心地解析ノートブック (V3/V4/V5 プロ仕様)
このノートブックでは、ユーザーが記録した「フラグ（不快ボタン）なし」の走行データを解析し、既存の予測モデルへの寄与度（AUC向上）を検証します。

## 実行ステップ
1. Google Driveのマウント（Colabの場合）
2. データの自動取得（新旧GAS URL）
3. ISO 2631標準に基づく高度な特徴量抽出
4. **疑似ラベリング（Pseudo-Labeling）**: 確率分布に基づき、相対的に「安全」「不快」な区間を抽出
5. 再学習と既存検証セットに対するAUC比較検証

In [ ]:
# 1. 環境構築 (Colabの場合)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # プロジェクトディレクトリに移動 (各自のパスに合わせて調整してください)
    # %cd /content/drive/MyDrive/ambulance_analysis
    print("Google Drive mounted.")
except ImportError:
    print("Local environment detected.")

!pip install optuna shap japanize-matplotlib joblib lightgbm

In [ ]:
import pandas as pd
import numpy as np
import requests
import os
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from datetime import datetime
import shutil
from scipy import signal
from sklearn.metrics import roc_curve, auc, roc_auc_score, precision_recall_curve, confusion_matrix
from sklearn.model_selection import GroupKFold
import lightgbm as lgb
import japanize_matplotlib

BASIC_GAS_URL = "https://script.google.com/macros/s/AKfycbyza-BCowCNcWYb-63gx1gd4UARcYTeJ8DXqv-rrZwcRryWqfZanAnXfyrf6jFxMEfDIA/exec"
NEW_GAS_URL = "https://script.google.com/macros/s/AKfycbyDUe-9y4oDoZYq-Cttk7fiGJohPwyr7LiFMijwZvZaThDMAcG0d7AQnndsnYPDBbrw9A/exec"
MODEL_DIR = "models"


## 2. データの取得と前処理

In [ ]:
def apply_iso_2631_weighting(series, axis='z', fs=50.0):
    nyquist = fs / 2
    # nyquist がフィルタ周波数（0.5Hz）以下の場合は適用不能
    if len(series) < 10 or nyquist <= 0.5: return series
    
    try:
        if axis == 'z': # Wk (Vertical)
            low = 0.5 / nyquist
            # 20Hz(nyquist=10Hz)だと15Hzはフィルタ不可なので、サンプリングに合わせて上限を制限
            high = min(15.0, nyquist * 0.9) / nyquist
            if low >= high: return series
            b, a = signal.butter(2, [low, high], btype='bandpass')
        else: # Wd (Horizontal)
            low = 0.4 / nyquist
            high = min(5.0, nyquist * 0.9) / nyquist
            if low >= high: return series
            b, a = signal.butter(2, [low, high], btype='bandpass')
        return signal.lfilter(b, a, series)
    except Exception:
        return series

def calculate_vdv(accel_series, window=50):
    return (accel_series**4).rolling(window=window, center=True).mean()**(1/4)

def preprocess_ride_data(df):
    if df.empty: return df
    cols = ["time_ms", "rawG_X", "rawG_Y", "rawG_Z", "jerk_X", "jerk_Y", "jerk_Z", "speed_kmh", "uncomfortable", "age"]
    for col in cols:
        if col in df.columns: 
            df[col] = pd.to_numeric(df[col], errors='coerce')
        elif col == "age":
            df["age"] = 40.0
            
    df = df.dropna(subset=["time_ms", "rawG_Z"]).sort_values("time_ms").reset_index(drop=True)
    if df.empty: return df
    
    df["time_gap"] = df["time_ms"].diff().fillna(pd.NA)
    df["ride_id"] = (df["time_gap"] > 60000).cumsum()
    
    # 各走行（ride_id）ごとにHzを推定し、動的にパラメータを調整する救済ロジック
    def process_ride(group):
        dt_ms = group["time_ms"].diff().median()
        hz = 1000.0 / dt_ms if not (pd.isna(dt_ms) or dt_ms <= 0) else 50.0
        
        # 窓幅の計算
        win_smooth = max(1, int(10 * (hz/50.0)))
        win_1s = max(1, int(hz))
        win_3s = max(1, int(3 * hz))
        
        group["rawG_X_smooth"] = group["rawG_X"].rolling(win_smooth, center=True).mean()
        group["rawG_Y_smooth"] = group["rawG_Y"].rolling(win_smooth, center=True).mean()
        group["jerk_Z_smooth"] = group["jerk_Z"].rolling(win_smooth, center=True).mean()
        
        # ISO 2631 重み付け (hz を渡す)
        group["iso_G_Z"] = apply_iso_2631_weighting(group["rawG_Z"], 'z', hz)
        group["iso_G_X"] = apply_iso_2631_weighting(group["rawG_X"], 'xy', hz)
        group["iso_G_Y"] = apply_iso_2631_weighting(group["rawG_Y"], 'xy', hz)
        
        group["VDV_Z"] = (group["iso_G_Z"]**4).rolling(window=win_1s, center=True).mean()**(1/4)
        group["jerk_Z_std_1s"] = group["jerk_Z_smooth"].rolling(win_1s, center=True).std()
        group["max_jerk_Z_3s"] = group["jerk_Z"].rolling(win_3s, min_periods=1).max()
        group["total_G_XY"] = np.sqrt(group["rawG_X_smooth"]**2 + group["rawG_Y_smooth"]**2)
        group["max_total_G_XY_3s"] = group["total_G_XY"].rolling(win_3s, min_periods=1).max()
        
        if "uncomfortable" in group.columns and group["uncomfortable"].sum() > 0:
            shift_start = int(1.5 * hz)
            shift_end = int(0.2 * hz)
            win_label = max(1, shift_start - shift_end)
            group["target"] = group["uncomfortable"].shift(-shift_start).rolling(window=win_label, min_periods=1).max().fillna(0)
        else:
            group["target"] = 0
        return group

    df = df.groupby("ride_id", group_keys=False).apply(process_ride)
    return df.dropna(subset=["iso_G_Z", "total_G_XY"]).reset_index(drop=True)

In [ ]:
# 変数の初期化（エラー回避用）
df_labeled = pd.DataFrame()
df_flagless = pd.DataFrame()

try:
    print("教師データ（ラベルあり）を取得中...")
    res_labeled = requests.get(BASIC_GAS_URL, timeout=30)
    if res_labeled.status_code == 200:
        df_labeled_raw = pd.DataFrame(res_labeled.json())
        df_labeled = preprocess_ride_data(df_labeled_raw)
        print(f"教師データ: {df_labeled.shape[0]} samples")
    else:
        print(f"⚠️ 教師データの取得に失敗しました (Status: {res_labeled.status_code})")

    print("今回のフラグなしデータを取得中...")
    res_flagless = requests.get(NEW_GAS_URL, timeout=30)
    if res_flagless.status_code == 200:
        df_flagless_raw = pd.DataFrame(res_flagless.json())
        df_flagless = preprocess_ride_data(df_flagless_raw)
        print(f"フラグなしデータ: {df_flagless.shape[0]} samples")
    else:
        print(f"⚠️ フラグなしデータの取得に失敗しました (Status: {res_flagless.status_code})")
except Exception as e:
    print(f"データ取得/前処理エラー: {e}")

## 3. モデルロードと特徴量の自動合わせ込み

In [ ]:
# モデルの検索とロード
models = []
for i in range(10):
    paths = [
        f"{MODEL_DIR}/enhanced_lgb_seed_{i}.joblib", 
        f"./models/enhanced_lgb_seed_{i}.joblib",
        f"/content/drive/MyDrive/ambulance_analysis/models/enhanced_lgb_seed_{i}.joblib"
    ]
    for path in paths:
        if os.path.exists(path):
            models.append(joblib.load(path))
            break

if not models:
    print("❌ エラー: モデルファイル (.joblib) が見つかりません。")
    print("Google Driveがマウントされているか、作業ディレクトリに models フォルダがあるか確認してください。")
    features = []
    if not df_flagless.empty: df_flagless["pred_prob"] = np.nan
elif df_flagless.empty:
    print("⚠️ 解析対象のデータ (df_flagless) が空です。")
    features = []
else:
    # モデルから実際の特徴量名を取得
    features = models[0].feature_name()
    print(f"✅ {len(models)} 個のモデルをロードしました。")
    print(f"モデルが使用する特徴量: {features}")
    
    # 特徴量の存在確認 & 欠損補完
    for f in features:
        if not df_labeled.empty and f not in df_labeled.columns: df_labeled[f] = 0
        if f not in df_flagless.columns: df_flagless[f] = 0

    if "age" in df_flagless.columns:
        median_age = df_labeled["age"].median() if not df_labeled.empty else 40.0
        df_flagless["age"] = df_flagless["age"].fillna(median_age)
    if not df_labeled.empty and "age" in df_labeled.columns:
        df_labeled["age"] = df_labeled["age"].fillna(df_labeled["age"].median())

    probs = np.mean([model.predict(df_flagless[features]) for model in models], axis=0)
    df_flagless["pred_prob"] = probs
    print("\n推論完了。不快確率の分布:")
    print(df_flagless["pred_prob"].describe())

In [ ]:
if df_flagless.empty or "pred_prob" not in df_flagless.columns or df_flagless["pred_prob"].isna().all():
    print("⚠️ 推論が行われていないため、以降の解析をスキップします。")
else:
    Q_LOW = df_flagless["pred_prob"].quantile(0.10)
    Q_HIGH = df_flagless["pred_prob"].quantile(0.90)

    df_flagless["pseudo_label"] = np.nan
    df_flagless.loc[df_flagless["pred_prob"] <= Q_LOW, "pseudo_label"] = 0
    df_flagless.loc[df_flagless["pred_prob"] >= Q_HIGH, "pseudo_label"] = 1

    pseudo_data = df_flagless.dropna(subset=["pseudo_label"]).copy()
    print(f"疑似ラベル付与数: {len(pseudo_data)}")
    print(f"  - 正常(0)採用閾値: {Q_LOW:.3f}")
    print(f"  - 不快(1)採用閾値: {Q_HIGH:.3f}")

## 4. AUC向上検証（再学習実験）

In [ ]:
if 'pseudo_data' not in locals() or pseudo_data.empty or df_labeled.empty or not features:
    print("⚠️ 比較に必要なデータ（教師データまたは疑似ラベル）が不足しています。")
    base_auc = 0.5
    exp_auc = 0.5
else:
    def evaluate_auc(X_train, y_train, X_test, y_test):
        params = {"objective": "binary", "metric": "auc", "verbosity": -1, "is_unbalance": True, "seed": 42}
        dtrain = lgb.Dataset(X_train, label=y_train)
        model = lgb.train(params, dtrain)
        return roc_auc_score(y_test, model.predict(X_test))

    gkf = GroupKFold(n_splits=5)
    try: 
        train_idx, test_idx = next(gkf.split(df_labeled[features], df_labeled["target"], df_labeled["ride_id"]))
        df_train_orig = df_labeled.iloc[train_idx]; df_test = df_labeled.iloc[test_idx]

        base_auc = evaluate_auc(df_train_orig[features], df_train_orig["target"], df_test[features], df_test["target"])
        X_train_exp = pd.concat([df_train_orig[features], pseudo_data[features]])
        y_train_exp = pd.concat([df_train_orig["target"], pseudo_data["pseudo_label"]])
        exp_auc = evaluate_auc(X_train_exp, y_train_exp, df_test[features], df_test["target"])

        print(f"ベースライン AUC: {base_auc:.5f}")
        print(f"追加学習後 AUC: {exp_auc:.5f}")
        print(f"精度向上率: {exp_auc - base_auc:+.5f}")
    except Exception as e:
        print(f"検証エラー: {e}")
        base_auc = 0.5
        exp_auc = 0.5

## 5. レポート生成と全図表のダウンロード

In [ ]:
out_dir = f"flagless_analysis_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(out_dir, exist_ok=True)
print(f"📦 出力ディレクトリを作成しました: {out_dir}")

if not df_flagless.empty and "pred_prob" in df_flagless.columns and not df_flagless["pred_prob"].dropna().empty:
    # 1. 時系列推移グラフ
    plt.figure(figsize=(15, 6))
    t_min = (df_flagless["time_ms"] - df_flagless["time_ms"].min()) / 60000
    plt.plot(t_min, df_flagless["pred_prob"], label="予測不快確率", color="crimson", lw=1.5)
    plt.fill_between(t_min, 0, df_flagless["pred_prob"], color="crimson", alpha=0.15)
    plt.axhline(0.5, color="gray", linestyle="--", label="不快境界(0.5)")
    plt.ylim(0, 1.0); plt.xlabel("経過時間 [分]"); plt.ylabel("不快確率"); plt.title("走行データの不快予測推移"); plt.legend(); plt.grid(True, alpha=0.2)
    plt.savefig(f"{out_dir}/01_time_series_prob.png", bbox_inches="tight")
    plt.show()

    # 2. 確率分布ヒストグラム
    plt.figure(figsize=(8, 6))
    sns.histplot(df_flagless["pred_prob"], bins=30, kde=True, color="salmon")
    if 'Q_LOW' in locals(): plt.axvline(Q_LOW, color="green", linestyle="--", label=f"正常境界({Q_LOW:.2f})")
    if 'Q_HIGH' in locals(): plt.axvline(Q_HIGH, color="red", linestyle="--", label=f"不快境界({Q_HIGH:.2f})")
    plt.title("不快確率の分布と疑似ラベル閾値"); plt.xlabel("推論確率"); plt.legend()
    plt.savefig(f"{out_dir}/02_prob_distribution.png", bbox_inches="tight")
    plt.show()

    # 3. AUC向上比較（棒グラフ）
    plt.figure(figsize=(6, 5))
    plt.bar(["Baseline", "With Flagless"], [base_auc, exp_auc], color=["gray", "royalblue"])
    plt.ylim(0.4, 1.0); plt.ylabel("AUC"); plt.title("疑似ラベル追加による精度変化")
    for i, v in enumerate([base_auc, exp_auc]):
        plt.text(i, v + 0.01, f"{v:.4f}", ha="center", fontweight="bold")
    plt.savefig(f"{out_dir}/03_auc_comparison.png", bbox_inches="tight")
    plt.show()

    # 4. Markdownレポート生成
    with open(f"{out_dir}/analysis_summary.md", "w", encoding="utf-8") as f:
        f.write("# フラグなし（自動記録）データ解析レポート\n\n")
        f.write(f"**分析実施日:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write("## 1. 精度向上検証結果\n")
        f.write(f"- **ベースライン AUC (既存データのみ):** {base_auc:.5f}\n")
        f.write(f"- **追加学習後 AUC (フラグなし追加):** {exp_auc:.5f}\n")
        f.write(f"- **精度変化:** {exp_auc - base_auc:+.5f}\n\n")
        f.write("## 2. 疑似ラベリング統計\n")
        f.write(f"- 採用されたサンプル数: {len(pseudo_data) if 'pseudo_data' in locals() else 0} 件\n")
        if 'Q_HIGH' in locals(): f.write(f"- 不快(1)判定の閾値: {Q_HIGH:.3f} 以上\n")
        if 'Q_LOW' in locals(): f.write(f"- 正常(0)判定の閾値: {Q_LOW:.3f} 以下\n\n")
        f.write("## 3. 可視化チャート構成\n")
        f.write("1. **01_time_series_prob.png**: 走行時間軸に沿った不快確率の推移\n")
        f.write("2. **02_prob_distribution.png**: 推論確率の全体分布とラベル付け境界\n")
        f.write("3. **03_auc_comparison.png**: 既存モデルに対する寄与度の比較\n")

    # 5. ZIP化と一括ダウンロード（Colab環境用）
    try:
        from google.colab import files
        zip_name = out_dir
        shutil.make_archive(zip_name, 'zip', out_dir)
        files.download(f"{zip_name}.zip")
        print(f"\n✅ 成果物を一括ZIP化してダウンロードを開始しました: {zip_name}.zip")
    except ImportError:
        print(f"\n📁 ローカル環境のため、{out_dir}/ フォルダに保存しました。")
else:
    print("解析データが不足しているため、レポート生成をスキップしました。")